### Storing only MAT files

In [ ]:
input_path = 'NAMES.txt'     
output_path = 'mat_files.txt' 

mat_filenames = []

with open(input_path, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line.endswith('.mat'):
            mat_filenames.append(line)

with open(output_path, 'w', encoding='utf-8') as out:
    for filename in mat_filenames:
        out.write(filename + '\n')

print(f"Extracted {len(mat_filenames)} .mat filenames to '{output_path}'")

Extracted 276 .mat filenames to 'mat_files.txt'


### Dataload to CSV

In [ ]:
import h5py
import pandas as pd
import numpy as np
import os

mat_list_path = 'mat_files.txt'

with open(mat_list_path, 'r', encoding='utf-8') as f:
    mat_files = [line.strip() for line in f if line.strip().endswith('.mat')]

for mat_filename in mat_files:
    input_file = os.path.join('epbin-data_DIGNUMWORDS-SHAPE', mat_filename)
    print(f"Trying to load: {input_file}")

    try:
        with h5py.File(input_file, 'r') as f:
            print(f"Keys in {mat_filename}: {list(f.keys())}")

            if 'data' in f:
                data = f['data'][:]
            else:
                print(f"Key 'data' not found in {mat_filename}, skipping.")
                continue

        
        data = np.squeeze(data)
        if data.ndim != 3:
            print(f"Unexpected shape after squeeze: {data.shape}, skipping.")
            continue

        time_len, chan_len, cond_len = data.shape 

        time = np.repeat(np.arange(time_len), chan_len * cond_len)
        channel = np.tile(np.repeat(np.arange(chan_len), cond_len), time_len)
        condition = np.tile(np.arange(cond_len), time_len * chan_len)

        flattened_data = data.reshape(-1, order='F') 

        assert len(time) == len(channel) == len(condition) == len(flattened_data), \
            f"Length mismatch: time={len(time)}, channel={len(channel)}, condition={len(condition)}, data={len(flattened_data)}"

        df_long = pd.DataFrame({
            'time': time,
            'channel': channel,
            'condition': condition,
            'value': flattened_data
        })

        safe_filename = mat_filename.replace(' ', '_').replace('.mat', '.csv')
        output_path = os.path.join('data', safe_filename)
        os.makedirs('data', exist_ok=True)
        df_long.to_csv(output_path, index=False)
        print(f"Saved CSV to: {output_path}\n")

    except Exception as e:
        print(f"Error processing {mat_filename}: {e}")

Trying to load: epbin-data_DIGNUMWORDS-SHAPE/epbin_Dig_1F_C1 21 rr fixAF7 fixF7 icfilt ica ep1 but chanlocs chansel chanlabels S19.mat
Keys in epbin_Dig_1F_C1 21 rr fixAF7 fixF7 icfilt ica ep1 but chanlocs chansel chanlabels S19.mat: ['data']
Saved CSV to: data/epbin_Dig_1F_C1_21_rr_fixAF7_fixF7_icfilt_ica_ep1_but_chanlocs_chansel_chanlabels_S19.csv

Trying to load: epbin-data_DIGNUMWORDS-SHAPE/epbin_Dig_1F_C1 21 rr fixAF7 icfilt ica ep1 but chanlocs chansel chanlabels S09.mat
Keys in epbin_Dig_1F_C1 21 rr fixAF7 icfilt ica ep1 but chanlocs chansel chanlabels S09.mat: ['data']
Saved CSV to: data/epbin_Dig_1F_C1_21_rr_fixAF7_icfilt_ica_ep1_but_chanlocs_chansel_chanlabels_S09.csv

Trying to load: epbin-data_DIGNUMWORDS-SHAPE/epbin_Dig_1F_C1 21 rr fixAF8 fixC5 fixFT7 ica ep1 but chanlocs chansel chanlabels S15.mat
Keys in epbin_Dig_1F_C1 21 rr fixAF8 fixC5 fixFT7 ica ep1 but chanlocs chansel chanlabels S15.mat: ['data']
Saved CSV to: data/epbin_Dig_1F_C1_21_rr_fixAF8_fixC5_fixFT7_ica_ep1_